# LOH.1 - Green's functions and manual convolution

This notebook opens up what `ShakerMaker.run()` does internally, on the SCEC
LOH.1 model with three surface receivers at `(8,8)`, `(6,8)` and `(4,4)` km.

We:

1. build the LOH.1 crust + a Gaussian double-couple source,
2. run the engine with `save_gf=True` so each station keeps its **Green's
   functions**,
3. read the **nine elementary Green's functions** (the `tdata` tensor) and the
   recombined `(Z, E, N)` impulse response,
4. reproduce the seismogram **by hand**, convolving the impulse response with
   the source time function (exactly what `run()` does internally), and
5. check that the manual convolution matches `get_response()`.

Every figure is saved as a `.png` in this folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from shakermaker import shakermaker
from shakermaker.crustmodel import CrustModel
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions.gaussian import Gaussian

## 1. Crust model (SCEC LOH.1)

A 1 km slow layer over a half-space, essentially elastic (very high Q).

In [ ]:
crust = CrustModel(2)
crust.add_layer(1.0, 4.000, 2.000, 2.600, 10000., 10000.)   # slow layer (1 km)
crust.add_layer(0.0, 6.000, 3.464, 2.700, 10000., 10000.)   # half-space
print(crust)

crust.plot()
plt.gcf().savefig("loh1_crust_layers.png", dpi=150, bbox_inches="tight")

crust.plot_profile()
plt.gcf().savefig("loh1_velocity_profile.png", dpi=150, bbox_inches="tight")

## 2. Source and source time function

A vertical strike-slip double couple at 2 km depth. The source time function is
a Gaussian; its seismic moment is `M0`.

We keep a **unit** STF (`M0 = 1`) in `stf` for the manual convolution later, and
scale by `M0` separately. Because convolution is linear in the STF,
`stf(M0=1).convolve(...) * M0` is identical to convolving with a Gaussian that
carries `M0` directly - which is what the source uses inside `run()`.

In [ ]:
dt = 0.005
sigma = 0.06
t0 = 6 * sigma
M0 = 1e18 / 5e14 / 2

# Source STF (carries M0) used by the engine
source = PointSource([0, 0, 2.0], [0., 90., 0.],
                     stf=Gaussian(t0=t0, freq=1/sigma, M0=M0, derivative=False))
fault = FaultSource([source], metadata={"name": "LOH1_source"})

# Unit STF for the manual convolution (M0 applied separately)
stf = Gaussian(t0=t0, freq=1/sigma, M0=1, derivative=False)
stf.dt = dt
t_stf = stf.t
gauss_SM = stf.data

fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(t_stf, gauss_SM, color="tab:red", lw=1.6)
ax.set_xlabel("time (s)"); ax.set_ylabel("amplitude")
ax.set_title("Gaussian source time function (unit M0)")
ax.set_xlim(0, 1.0); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("loh1_stf.png", dpi=150, bbox_inches="tight")

## 3. Stations

Three surface receivers. `save_gf=True` tells the engine to keep each station's
Green's functions in memory (off by default).

In [ ]:
s1 = Station([8.0, 8.0, 0.0], metadata={"name": "sta01", "save_gf": True})
s2 = Station([6.0, 8.0, 0.0], metadata={"name": "sta02", "save_gf": True})
s3 = Station([4.0, 4.0, 0.5], metadata={"name": "sta03", "save_gf": True})
stations = StationList([s1, s2, s3], {})

## 4. Run

`check_parameters` reports the derived FK numbers first (pure arithmetic, no FK
run), then we run the three pairs.

In [ ]:
nfft, tb, dk = 2048 * 2, 20, 0.05 / 2

model = shakermaker.ShakerMaker(crust, fault, stations)
model.check_parameters(dt=dt, nfft=nfft, dk=dk, tb=tb, tmax=100)
model.run(dt=dt, nfft=nfft, tb=tb, dk=dk, smth=1)

## 5. Station responses (velocity)

`get_response()` returns the three-component velocity `(Z, E, N)` and the time
vector. Rows are components, columns are the three stations.

In [ ]:
resp = {
    "S1 (8,8,0)": s1.get_response(),
    "S2 (6,8,0)": s2.get_response(),
    "S3 (4,4,0.5)": s3.get_response(),
}

fig, axes = plt.subplots(3, 3, figsize=(12, 7), sharex=True)
comp_colors = {"Z": "tab:blue", "E": "tab:red", "N": "tab:green"}
for col, (name, (z, e, n, t)) in enumerate(resp.items()):
    for row, (lab, data) in enumerate(zip(["Z", "E", "N"], [z, e, n])):
        ax = axes[row, col]
        ax.plot(t, data, color=comp_colors[lab], lw=1.0)
        ax.set_xlim(0, 15); ax.grid(alpha=0.3)
        if row == 0:
            ax.set_title(name, fontweight="bold")
        if col == 0:
            ax.set_ylabel(rf"$\dot{{u}}_{lab}$")
for ax in axes[-1, :]:
    ax.set_xlabel("time (s)")
fig.suptitle("LOH.1 velocity responses", fontweight="bold")
fig.tight_layout()
fig.savefig("loh1_responses.png", dpi=150, bbox_inches="tight")

## 6. Green's functions

With `save_gf=True`, each station exposes `get_greens_functions()` -> a dict
keyed by source/subfault index. Each entry is
`(z, e, n, t, tdata, t0)` where:

- `z, e, n` are the **recombined impulse response** in (Z, E, N) for the
  source mechanism (before the source time function and before `M0`),
- `tdata` has shape `(1, 9, nt)`: the **nine elementary Green's functions**
  (Helmberger's DD/DS/SS set) for this source-receiver pair,
- `t0` is the reduction time and `t` the time vector.

In [ ]:
gf_s1 = s1.get_greens_functions()
gf_s2 = s2.get_greens_functions()
gf_s3 = s3.get_greens_functions()

z, e, n, t, tdata, t0 = gf_s1[0]
print("tdata shape:", tdata.shape, " (1, 9, nt)")
nt = tdata.shape[2]
t_gf = np.arange(nt) * (t[1] - t[0]) + t0

### The nine elementary Green's functions (station S1)

The `tdata` tensor: 9 traces that, recombined with the moment tensor and the
station azimuth, give the (Z, E, N) response.

In [ ]:
labels = [['Gf_11', 'Gf_12', 'Gf_13'],
          ['Gf_21', 'Gf_22', 'Gf_23'],
          ['Gf_31', 'Gf_32', 'Gf_33']]

fig, axes = plt.subplots(3, 3, figsize=(9, 6), sharex=True)
for i in range(3):
    for j in range(3):
        axes[i, j].plot(t_gf, tdata[0, i*3 + j, :], 'b-', lw=0.8)
        axes[i, j].set_title(labels[i][j])
        axes[i, j].grid(True, alpha=0.3)
        axes[i, j].set_xlim(0, 9)
fig.suptitle("S1 - nine elementary Green's functions (tdata)", fontweight="bold")
fig.tight_layout()
fig.savefig("loh1_gf_tensor.png", dpi=150, bbox_inches="tight")

### Recombined impulse response (Z, E, N)

The `z, e, n` arrays: the impulse response already recombined for the source
mechanism and rotated to the geographic components - still before the source
time function.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 6), sharex=True)
for ax, data, lab, color in zip(axes, [z, e, n], ['Z', 'E', 'N'], ['b', 'r', 'g']):
    ax.plot(t, data, color=color, lw=1.0)
    ax.set_title(f'Green function - {lab}', fontweight='bold')
    ax.set_ylabel('amplitude')
    ax.set_xlim(0, 9); ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('time (s)')
fig.tight_layout()
fig.savefig("loh1_gf_components.png", dpi=150, bbox_inches="tight")

## 7. Manual convolution = the seismogram

The seismogram is the impulse response convolved with the source time function
and scaled by the seismic moment. This is exactly what `run()` does internally
(`shakermaker.py`: `z_stf = psource.stf.convolve(z, t)`).

In [ ]:
z, e, n, t, tdata, t0 = gf_s1[0]

stf.dt = dt                     # unit Gaussian, sampled at the signal dt
z_stf = stf.convolve(z, t) * M0
e_stf = stf.convolve(e, t) * M0
n_stf = stf.convolve(n, t) * M0

fig, axes = plt.subplots(4, 1, figsize=(10, 9))
axes[0].plot(t_stf, gauss_SM * M0, 'k-', lw=1.5)
axes[0].set_title('Source time function (x M0)', fontweight='bold')
axes[0].set_xlim(0, 15); axes[0].grid(True, alpha=0.3)
for ax, data, lab, color in zip(axes[1:], [z_stf, e_stf, n_stf], ['Z', 'E', 'N'], ['b', 'r', 'g']):
    ax.plot(t, data, color=color, lw=1.0)
    ax.set_title(f'Seismogram - {lab}', fontweight='bold')
    ax.set_ylabel('velocity')
    ax.set_xlim(0, 15); ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('time (s)')
fig.tight_layout()
fig.savefig("loh1_seismograms_manual.png", dpi=150, bbox_inches="tight")

## 8. Check: manual convolution == `run()` output

The manual seismogram (impulse response convolved by hand) must match the
station response that `run()` produced. We overlay them for station S1.

In [ ]:
z_v1, e_v1, n_v1, t1 = s1.get_response()

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, manual, run_out, lab in zip(axes, [z_stf, e_stf, n_stf],
                                    [z_v1, e_v1, n_v1], ['Z', 'E', 'N']):
    ax.plot(t1, run_out, 'k-', lw=2.2, alpha=0.5, label='run() get_response')
    ax.plot(t, manual, 'r-', lw=1.0, label='manual convolution')
    ax.set_title(f'Component {lab}', fontweight='bold')
    ax.set_ylabel('velocity'); ax.set_xlim(0, 15); ax.grid(alpha=0.3)
    ax.legend(loc='upper right')
axes[-1].set_xlabel('time (s)')
fig.suptitle('Manual convolution vs run() - station S1', fontweight='bold')
fig.tight_layout()
fig.savefig("loh1_convolution_check.png", dpi=150, bbox_inches="tight")

# numeric check: interpolate run() onto the manual grid and correlate
zi = np.interp(t, t1, z_v1)
mask = (t >= 0) & (t <= 15)
cc = np.corrcoef(zi[mask], z_stf[mask])[0, 1]
print(f"Z correlation (manual vs run): {cc:.4f}")